# 18 — fato_contratos (Gold)

## Purpose
Build the `fato_contratos` fact table by combining `silver_compras` and `silver_pncp`,
joining with `dim_fornecedor` (temporal join per ADR-003) and `dim_tempo`.

## Grain
One row per contract: `cnpj_fornecedor + numero_contrato + data_referencia + fonte`

## Input
- `local/data/silver/silver_compras/` — partitioned by _year_month
- `local/data/silver/silver_pncp/data.parquet`
- `local/data/gold/dim_fornecedor.parquet`
- `local/data/gold/dim_tempo.parquet`

## Output
- `local/data/gold/fato_contratos.parquet`

## Steps
1. Imports and configuration
2. Load sources as lazy views
3. Build unified contract base (compras + pncp)
4. Join with dim_fornecedor (temporal join + current fallback)
5. Join with dim_tempo
6. Write to Parquet
7. Validation

## Architecture Notes
- **Grain:** one row per contract per source system.
- **data_referencia:** `data_vigencia_inicial` for compras (no data_assinatura available),
  `data_assinatura` for pncp. Used for temporal join and date_id.
- **Temporal join (ADR-003):** `d.valid_from <= data_referencia AND d.valid_to > data_referencia`.
  On first load all dim rows have valid_from = today — historical contracts will not match
  the temporal predicate and fall through to the current fallback.
- **_join_type:** `'temporal'` when temporal join succeeded, `'current_fallback'` when only
  is_current=True matched, `'unmatched'` when supplier not found in dim at all.
  This field tracks join quality and will evolve as SCD2 history accumulates.
- **contrato_sk:** MD5(cnpj_fornecedor || numero_contrato || data_referencia || fonte) —
  deterministic, idempotent, no global sort required.
- Idempotent — safe to re-run (overwrites output file).


## Step 1 — Imports and configuration

In [ ]:
import sys
import duckdb
from pathlib import Path
from datetime import datetime, timezone

# --- Resolve project root ---
def _find_root(start: Path) -> Path:
    current = start if start.is_dir() else start.parent
    for candidate in [current, *current.parents]:
        if (candidate / "utils" / "paths.py").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return start

try:
    _seed = Path(__vsc_ipynb_file__).resolve()
except NameError:
    _seed = Path.cwd().resolve()

PROJECT_ROOT = _find_root(_seed)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import silver_path, gold_path, ensure_dir

# --- Paths ---
SILVER_COMPRAS  = silver_path(PROJECT_ROOT, "silver_compras", glob=True)
SILVER_PNCP     = silver_path(PROJECT_ROOT, "silver_pncp")
DIM_FORNECEDOR  = gold_path(PROJECT_ROOT, "dim_fornecedor")
DIM_TEMPO       = gold_path(PROJECT_ROOT, "dim_tempo")
OUTPUT_PATH     = Path(gold_path(PROJECT_ROOT, "fato_contratos"))
OUTPUT_PATH_STR = gold_path(PROJECT_ROOT, "fato_contratos")
ensure_dir(OUTPUT_PATH.parent)

LOADED_AT = datetime.now(timezone.utc).isoformat()

print(f"Silver compras : {SILVER_COMPRAS}")
print(f"Silver pncp    : {SILVER_PNCP}")
print(f"Dim fornecedor : {DIM_FORNECEDOR}")
print(f"Dim tempo      : {DIM_TEMPO}")
print(f"Output         : {OUTPUT_PATH}")
print(f"loaded_at      : {LOADED_AT}")

## Step 2 — Load sources as lazy views

In [ ]:
con = duckdb.connect()
con.execute("SET threads TO 4")
con.execute("SET memory_limit = '8GB'")
con.execute("SET preserve_insertion_order = false")

con.execute(f"""
    CREATE OR REPLACE VIEW v_compras AS
    SELECT * FROM read_parquet('{SILVER_COMPRAS}', hive_partitioning=true)
""")

con.execute(f"""
    CREATE OR REPLACE VIEW v_pncp AS
    SELECT * FROM read_parquet('{SILVER_PNCP}')
""")

con.execute(f"""
    CREATE OR REPLACE VIEW v_dim_fornecedor AS
    SELECT * FROM read_parquet('{DIM_FORNECEDOR}')
""")

con.execute(f"""
    CREATE OR REPLACE VIEW v_dim_tempo AS
    SELECT * FROM read_parquet('{DIM_TEMPO}')
""")

counts = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM v_compras)        AS compras,
        (SELECT COUNT(*) FROM v_pncp)           AS pncp,
        (SELECT COUNT(*) FROM v_dim_fornecedor) AS dim_fornecedor,
        (SELECT COUNT(*) FROM v_dim_tempo)      AS dim_tempo
""").df()

print(counts.to_string(index=False))

## Step 3 — Build unified contract base (compras + pncp)

Harmonises the two sources into a single schema before joining with dimensions.

| Field | compras source | pncp source |
|---|---|---|
| data_referencia | data_vigencia_inicial | data_assinatura |
| numero_contrato | numero_contrato | numero_controle_pncp |
| cnpj_comprador | codigo_orgao (CNPJ not available) | orgao_cnpj |
| nome_comprador | nome_orgao | orgao_razao_social |
| valor_inicial | valor_global | valor_inicial |


In [ ]:
print("Building unified contract base...")
t0 = datetime.now()

con.execute("""
    CREATE OR REPLACE TABLE t_contratos_base AS

    WITH compras_raw AS (
        SELECT DISTINCT
            cnpj_normalized         AS cnpj_fornecedor,
            nome_fornecedor,
            NULL::VARCHAR           AS cnpj_comprador,
            codigo_orgao,
            nome_orgao              AS nome_comprador,
            codigo_unidade_gestora,
            nome_unidade_gestora,
            numero_contrato,
            numero_compra,
            objeto                  AS objeto_contrato,
            valor_global            AS valor_inicial,
            data_vigencia_inicial   AS data_referencia,
            data_vigencia_inicial,
            data_vigencia_final,
            _fonte                  AS fonte,
            _valor_negativo
        FROM v_compras
        WHERE cnpj_normalized IS NOT NULL
          AND data_vigencia_inicial IS NOT NULL
    ),
    -- When same contract has different objeto_contrato (source data quality issue),
    -- keep the row with the longest (most complete) description.
    -- When same contract has different data_vigencia_final (renewal),
    -- both rows are kept — data_vigencia_final is now part of the SK.
    compras_dedup AS (
        SELECT *
        FROM compras_raw
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY cnpj_fornecedor,
                         numero_contrato,
                         data_referencia,
                         codigo_unidade_gestora,
                         valor_inicial,
                         data_vigencia_final
            ORDER BY LENGTH(objeto_contrato) DESC
        ) = 1
    )
    SELECT * FROM compras_dedup

    UNION ALL

    SELECT
        cnpj_normalized                                         AS cnpj_fornecedor,
        nome_fornecedor,
        orgao_cnpj                                              AS cnpj_comprador,
        NULL::VARCHAR                                           AS codigo_orgao,
        orgao_razao_social                                      AS nome_comprador,
        unidade_codigo                                          AS codigo_unidade_gestora,
        unidade_nome                                            AS nome_unidade_gestora,
        COALESCE(numero_controle_pncp, numero_contrato_empenho) AS numero_contrato,
        NULL::VARCHAR                                           AS numero_compra,
        objeto                                                  AS objeto_contrato,
        valor_inicial,
        data_assinatura                                         AS data_referencia,
        data_vigencia_inicio                                    AS data_vigencia_inicial,
        data_vigencia_fim                                       AS data_vigencia_final,
        _fonte                                                  AS fonte,
        FALSE                                                   AS _valor_negativo
    FROM v_pncp
        WHERE cnpj_normalized IS NOT NULL
        AND data_assinatura IS NOT NULL
        AND data_assinatura BETWEEN '2021-01-01' AND '2030-12-31'
""")

n       = con.execute("SELECT COUNT(*) FROM t_contratos_base").fetchone()[0]
elapsed = (datetime.now() - t0).total_seconds()

dist = con.execute("""
    SELECT fonte, COUNT(*) AS total FROM t_contratos_base GROUP BY 1 ORDER BY 1
""").df()

print(f"t_contratos_base: {n:,} rows in {elapsed:.1f}s")
print()
print(dist.to_string(index=False))

## Step 4 — Join with dim_fornecedor (temporal join + current fallback)

### Join strategy (ADR-003)

**Temporal join (preferred):**
`d.valid_from <= c.data_referencia AND d.valid_to > c.data_referencia`

Links each contract to the supplier version that was active on the contract date.
On first load this will return 0 matches because all dim rows have valid_from = today.

**Current fallback:**
`d.is_current = True`

Used when temporal join yields no match. Captures the current supplier version.
Annotated as `_join_type = 'current_fallback'` for lineage tracking.

**Unmatched:**
`supplier_sk IS NULL` — supplier exists in contracts but not in dim_fornecedor.
Should be 0 or very close to 0 for known CNPJs.


In [ ]:
print("Joining with dim_fornecedor...")
t0 = datetime.now()

con.execute("""
    CREATE OR REPLACE TABLE t_contratos_with_supplier AS
    WITH temporal AS (
        -- Attempt 1: temporal join — correct per ADR-003
        SELECT
            c.*,
            d.supplier_sk                          AS supplier_sk,
            'temporal'                             AS _join_type
        FROM t_contratos_base c
        INNER JOIN v_dim_fornecedor d
            ON  d.cnpj_normalized = c.cnpj_fornecedor
            AND d.valid_from     <= c.data_referencia
            AND d.valid_to       >  c.data_referencia
    ),
    fallback AS (
        -- Attempt 2: current fallback — for contracts not matched temporally
        -- (expected on first load: all dim valid_from = today > historical dates)
        SELECT
            c.*,
            d.supplier_sk                          AS supplier_sk,
            'current_fallback'                     AS _join_type
        FROM t_contratos_base c
        LEFT JOIN temporal t
            ON t.cnpj_fornecedor = c.cnpj_fornecedor
            AND t.data_referencia = c.data_referencia
            AND t.fonte           = c.fonte
            AND t.numero_contrato = c.numero_contrato
        INNER JOIN v_dim_fornecedor d
            ON  d.cnpj_normalized = c.cnpj_fornecedor
            AND d.is_current      = TRUE
        WHERE t.cnpj_fornecedor IS NULL  -- not matched by temporal join
    ),
    unmatched AS (
        -- Contracts whose supplier is not in dim_fornecedor at all
        SELECT
            c.*,
            NULL::VARCHAR                          AS supplier_sk,
            'unmatched'                            AS _join_type
        FROM t_contratos_base c
        LEFT JOIN temporal  t ON t.cnpj_fornecedor = c.cnpj_fornecedor
            AND t.data_referencia = c.data_referencia
            AND t.fonte = c.fonte AND t.numero_contrato = c.numero_contrato
        LEFT JOIN fallback  f ON f.cnpj_fornecedor = c.cnpj_fornecedor
            AND f.data_referencia = c.data_referencia
            AND f.fonte = c.fonte AND f.numero_contrato = c.numero_contrato
        WHERE t.cnpj_fornecedor IS NULL
          AND f.cnpj_fornecedor IS NULL
    )
    SELECT * FROM temporal
    UNION ALL
    SELECT * FROM fallback
    UNION ALL
    SELECT * FROM unmatched
""")

n       = con.execute("SELECT COUNT(*) FROM t_contratos_with_supplier").fetchone()[0]
elapsed = (datetime.now() - t0).total_seconds()

join_dist = con.execute("""
    SELECT _join_type, COUNT(*) AS total
    FROM t_contratos_with_supplier
    GROUP BY 1 ORDER BY 2 DESC
""").df()

print(f"t_contratos_with_supplier: {n:,} rows in {elapsed:.1f}s")
print()
print(join_dist.to_string(index=False))

## Step 5 — Add dim_tempo FK and compute contrato_sk, write to Parquet

In [ ]:
print("Adding dim_tempo FK and writing final fato_contratos...")
t0 = datetime.now()

con.execute(f"""
    COPY (
        SELECT
            -- Surrogate key — grain: cnpj + contrato + data + unidade_gestora
            --                        + valor_inicial + data_vigencia_final + fonte
            -- data_vigencia_final included to handle contract renewals (same number, new end date)
            -- valor_inicial included to handle contract amendments (same number, updated value)
            md5(
                COALESCE(cnpj_fornecedor,                    '') || '|' ||
                COALESCE(numero_contrato,                     '') || '|' ||
                CAST(data_referencia AS VARCHAR)               || '|' ||
                COALESCE(fonte,                              '') || '|' ||
                COALESCE(codigo_unidade_gestora,             '') || '|' ||
                COALESCE(CAST(valor_inicial AS VARCHAR),     '') || '|' ||
                COALESCE(CAST(data_vigencia_final AS VARCHAR), '')
            )::VARCHAR                                          AS contrato_sk,

            -- Dimension FKs
            supplier_sk,
            CAST(strftime(data_referencia, '%Y%m%d') AS INTEGER) AS date_id,

            -- Natural keys
            cnpj_fornecedor,
            cnpj_comprador,
            codigo_orgao,
            nome_comprador,
            codigo_unidade_gestora,
            nome_unidade_gestora,
            nome_fornecedor,

            -- Contract attributes
            numero_contrato,
            numero_compra,
            objeto_contrato,
            fonte,
            _join_type,

            -- Measures
            valor_inicial,

            -- Dates
            data_referencia,
            data_vigencia_inicial,
            data_vigencia_final,

            -- Quality flags
            _valor_negativo,

            -- Audit
            '{LOADED_AT}'::TIMESTAMPTZ                          AS _loaded_at

        FROM t_contratos_with_supplier c
        LEFT JOIN v_dim_tempo t
            ON t.date_id = CAST(strftime(c.data_referencia, '%Y%m%d') AS INTEGER)

        ORDER BY data_referencia, cnpj_fornecedor
    ) TO '{OUTPUT_PATH_STR}' (FORMAT PARQUET)
""")

con.execute("DROP TABLE IF EXISTS t_contratos_base")
con.execute("DROP TABLE IF EXISTS t_contratos_with_supplier")

file_size_mb = OUTPUT_PATH.stat().st_size / 1_048_576
elapsed      = (datetime.now() - t0).total_seconds()
print(f"Written : {OUTPUT_PATH}")
print(f"Size    : {file_size_mb:.2f} MB")
print(f"Time    : {elapsed:.1f}s")

## Step 6 — Validation

In [ ]:
checks = []

total = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')"
).fetchone()[0]

# CHECK 1 — row count
# silver_compras rows that differ only in objeto_contrato are deduplicated
# via QUALIFY (keep longest description). Renewals (different data_vigencia_final)
# are kept as separate rows — both are valid analytical records.
n_compras_dedup = con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT
            cnpj_normalized, codigo_unidade_gestora, numero_contrato,
            valor_global, data_vigencia_inicial, data_vigencia_final, _fonte
        FROM read_parquet('{SILVER_COMPRAS}', hive_partitioning=true)
        WHERE cnpj_normalized IS NOT NULL AND data_vigencia_inicial IS NOT NULL
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY cnpj_normalized, codigo_unidade_gestora,
                         numero_contrato, valor_global,
                         data_vigencia_inicial, data_vigencia_final
            ORDER BY LENGTH(objeto) DESC
        ) = 1
    )
""").fetchone()[0]

n_pncp = con.execute(
    "SELECT COUNT(*) FROM v_pncp WHERE cnpj_normalized IS NOT NULL AND data_assinatura IS NOT NULL"
).fetchone()[0]

expected = n_compras_dedup + n_pncp
checks.append((
    "Row count = compras_dedup + pncp",
    total == expected,
    f"{total:,} (expected {expected:,} = {n_compras_dedup:,} dedup + {n_pncp:,})"
))

# CHECK 2 — contrato_sk unique
unique_sk = con.execute(
    f"SELECT COUNT(DISTINCT contrato_sk) FROM read_parquet('{OUTPUT_PATH_STR}')"
).fetchone()[0]
checks.append(("contrato_sk unique", unique_sk == total,
               f"{unique_sk:,} unique / {total:,} total"))

# CHECK 3 — no null contrato_sk
null_sk = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}') WHERE contrato_sk IS NULL"
).fetchone()[0]
checks.append(("contrato_sk not null", null_sk == 0, f"{null_sk} nulls"))

# CHECK 4 — fonte has exactly 2 values
fontes = sorted(con.execute(
    f"SELECT DISTINCT fonte FROM read_parquet('{OUTPUT_PATH_STR}')"
).df()["fonte"].tolist())
checks.append(("fonte has 2 values", len(fontes) == 2, str(fontes)))

# CHECK 5 — unmatched supplier_sk rate < 5%
unmatched = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE _join_type = 'unmatched'
""").fetchone()[0]
unmatched_pct = unmatched / total * 100 if total > 0 else 0
checks.append(("Unmatched supplier_sk < 5%", unmatched_pct < 5.0,
               f"{unmatched:,} ({unmatched_pct:.2f}%)"))

# CHECK 6 — date_id in dim_tempo range
out_of_range = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE date_id < 20210101 OR date_id > 20301231
""").fetchone()[0]
checks.append(("date_id in dim_tempo range", out_of_range == 0,
               f"{out_of_range} out-of-range dates"))

# CHECK 7 — no null date_id
null_date = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}') WHERE date_id IS NULL"
).fetchone()[0]
checks.append(("date_id not null", null_date == 0, f"{null_date} nulls"))

# CHECK 8 — no negative values flagged as valid
neg_unflagged = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{OUTPUT_PATH_STR}')
    WHERE valor_inicial < 0 AND _valor_negativo = false
""").fetchone()[0]
checks.append(("No unflagged negative values", neg_unflagged == 0,
               f"{neg_unflagged} unexpected negatives"))

# ── Report ────────────────────────────────────────────────────────────────────
print(f"\n{'CHECK':<50} {'STATUS':<8} DETAIL")
print("-" * 90)
all_pass = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    print(f"{name:<50} [{status}]   {detail}")
    if not passed:
        all_pass = False

print()
if all_pass:
    print("All checks PASSED — fato_contratos ready.")
else:
    print("One or more checks FAILED — review output above.")

# ── Join type distribution ────────────────────────────────────────────────────
print()
print("Join type distribution:")
jd = con.execute(f"""
    SELECT _join_type, COUNT(*) AS total,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('{OUTPUT_PATH_STR}')
    GROUP BY 1 ORDER BY 2 DESC
""").df()
print(jd.to_string(index=False))

# ── Value distribution by fonte ───────────────────────────────────────────────
print()
print("Value distribution by fonte:")
vd = con.execute(f"""
    SELECT
        fonte,
        COUNT(*)             AS contratos,
        SUM(valor_inicial)   AS valor_total,
        AVG(valor_inicial)   AS valor_medio,
        MAX(valor_inicial)   AS maior_contrato
    FROM read_parquet('{OUTPUT_PATH_STR}')
    GROUP BY 1 ORDER BY 1
""").df()
print(vd.to_string(index=False))

# ── Sample rows ───────────────────────────────────────────────────────────────
print()
print("Sample rows:")
s = con.execute(f"""
    SELECT contrato_sk, cnpj_fornecedor, nome_comprador,
           fonte, valor_inicial, data_referencia, _join_type
    FROM read_parquet('{OUTPUT_PATH_STR}')
    LIMIT 4
""").df()
print(s.to_string(index=False))